# Voting Ensemble Classifier — SPY & TSLA
Hard voting across SVM, Logistic Regression, Random Forest, and XGBoost using selected balanced parameters per model.

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

In [2]:
FEATURES = [
    'daily_return', 'weekly_return', 'ma_cross', 'dist_from_ma21', 'daily_range',
    'rsi_14', 'macd_hist', 'bb_position', 'volatility_7', 'volatility_20',
    'volume_change', 'volume_ratio',
    'lag_return_1', 'lag_return_3', 'lag_return_5',
    'month', 'is_earnings_week',
    'vix', 'is_major_event'
]
TARGET_CLS = 'target_direction'

def walk_forward_splits(df, train_window, test_window=42, embargo=5):
    splits = []
    n = len(df)
    start = 0
    while start + train_window + embargo + test_window <= n:
        train_idx = list(range(start, start + train_window))
        test_idx  = list(range(start + train_window + embargo,
                               start + train_window + embargo + test_window))
        splits.append((train_idx, test_idx))
        start += test_window
    return splits

def seasonal_breakdown(actual, pred, seasons):
    print(f"\nSeasonal Performance Breakdown:")
    print(f"{'Season':<8}  {'n':>5}  {'acc':>6}  {'f1':>6}  {'prec':>6}  {'rec':>6}")
    print("-" * 50)
    season_df = pd.DataFrame({'actual': actual, 'pred': pred, 'season': seasons})
    for season in ['Winter', 'Spring', 'Summer', 'Fall']:
        grp = season_df[season_df['season'] == season]
        if len(grp) == 0:
            continue
        acc  = accuracy_score(grp['actual'], grp['pred'])
        f1   = f1_score(grp['actual'], grp['pred'], zero_division=0)
        prec = precision_score(grp['actual'], grp['pred'], zero_division=0)
        rec  = recall_score(grp['actual'], grp['pred'], zero_division=0)
        print(f'{season:<8}  {len(grp):>5}  {acc:>6.3f}  {f1:>6.3f}  {prec:>6.3f}  {rec:>6.3f}')

## SPY — Voting Ensemble

| Model | Type | Key Parameters |
|---|---|---|
| SVM | Balanced | C=100, gamma=0.1 |
| LR | Balanced | C=0.001, l2 |
| RF | Balanced | n_est=200, depth=6, min_leaf=5, max_feat=sqrt |
| XGBoost | Balanced | n_est=500, depth=4, lr=0.01, subsample=0.8 |

In [3]:
spy = pd.read_csv('data/SPY_features.csv', parse_dates=['date'], index_col='date').sort_index()
spy_cls = spy[FEATURES + [TARGET_CLS] + ['season']].dropna().copy()
spy_cls_reset = spy_cls.reset_index()
spy_folds = walk_forward_splits(spy_cls, train_window=189)
print(f'SPY shape: {spy_cls.shape}, Folds: {len(spy_folds)}')

neg = (spy_cls[TARGET_CLS] == 0).sum()
pos = (spy_cls[TARGET_CLS] == 1).sum()
spy_scale_pos_weight = neg / pos
print(f'scale_pos_weight: {spy_scale_pos_weight:.3f}')

SPY shape: (2515, 21), Folds: 55
scale_pos_weight: 0.832


In [4]:
spy_actual, spy_pred, spy_seasons = [], [], []
fold_scores = []

for train_idx, test_idx in spy_folds:
    X_train = spy_cls.iloc[train_idx][FEATURES]
    y_train = spy_cls.iloc[train_idx][TARGET_CLS]
    X_test  = spy_cls.iloc[test_idx][FEATURES]
    y_test  = spy_cls.iloc[test_idx][TARGET_CLS]
    seasons = spy_cls_reset.iloc[test_idx]['season'].values

    # Scale for SVM and LR
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    # Individual models — note SVM and LR use scaled features
    svm = SVC(C=100, gamma=0.1, kernel='rbf', class_weight='balanced', random_state=42, probability=True)
    lr  = LogisticRegression(C=0.001, solver='lbfgs', class_weight='balanced', max_iter=1000, random_state=42)
    rf  = RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=5,
                                  max_features='sqrt', class_weight='balanced', random_state=42, n_jobs=-1)
    xgb = XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.01, subsample=0.8,
                         scale_pos_weight=spy_scale_pos_weight, objective='binary:logistic',
                         eval_metric='logloss', random_state=42, n_jobs=-1, verbosity=0)

    # Fit scaled models
    svm.fit(X_train_sc, y_train)
    lr.fit(X_train_sc, y_train)
    # Fit unscaled models
    rf.fit(X_train, y_train)
    xgb.fit(X_train, y_train)

    # Collect predictions from each
    svm_pred = svm.predict(X_test_sc)
    lr_pred  = lr.predict(X_test_sc)
    rf_pred  = rf.predict(X_test)
    xgb_pred = xgb.predict(X_test)

    # Hard voting — majority wins
    stacked = np.stack([svm_pred, lr_pred, rf_pred, xgb_pred], axis=1)
    y_p = (stacked.sum(axis=1) >= 3).astype(int)

    cm = confusion_matrix(y_test, y_p, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0

    fold_scores.append({
        'accuracy':    accuracy_score(y_test, y_p),
        'f1':          f1_score(y_test, y_p, zero_division=0),
        'precision':   precision_score(y_test, y_p, zero_division=0),
        'recall':      recall_score(y_test, y_p, zero_division=0),
        'specificity': spec
    })
    spy_actual.extend(y_test)
    spy_pred.extend(y_p)
    spy_seasons.extend(seasons)

print(classification_report(spy_actual, spy_pred, zero_division=0))
print(pd.DataFrame(fold_scores).describe())

              precision    recall  f1-score   support

           0       0.46      0.56      0.50      1038
           1       0.56      0.46      0.51      1272

    accuracy                           0.50      2310
   macro avg       0.51      0.51      0.50      2310
weighted avg       0.51      0.50      0.50      2310

        accuracy         f1  precision     recall  specificity
count  55.000000  55.000000  55.000000  55.000000    55.000000
mean    0.503896   0.477617   0.568413   0.462921     0.560184
std     0.079201   0.151713   0.131894   0.222881     0.225352
min     0.309524   0.071429   0.280000   0.040000     0.125000
25%     0.452381   0.338095   0.500000   0.269697     0.371711
50%     0.500000   0.488889   0.550000   0.434783     0.545455
75%     0.571429   0.577381   0.666667   0.637931     0.743421
max     0.642857   0.730159   0.909091   0.913043     0.944444


In [5]:
seasonal_breakdown(spy_actual, spy_pred, spy_seasons)


Seasonal Performance Breakdown:
Season        n     acc      f1    prec     rec
--------------------------------------------------
Winter      552   0.480   0.506   0.529   0.485
Spring      575   0.517   0.489   0.556   0.436
Summer      580   0.512   0.518   0.610   0.450
Fall        603   0.506   0.508   0.550   0.472


## TSLA — Voting Ensemble

| Model | Type | Key Parameters |
|---|---|---|
| SVM | Baseline | C=10, gamma=0.1 |
| LR | Best F1 | C=10, l1 |
| RF | Balanced | n_est=200, depth=4, min_leaf=10, max_feat=sqrt |
| XGBoost | Balanced | n_est=100, depth=3, lr=0.01, subsample=1.0 |

In [6]:
tsla = pd.read_csv('data/TSLA_features.csv', parse_dates=['date'], index_col='date').sort_index()
tsla_cls = tsla[FEATURES + [TARGET_CLS] + ['season']].dropna().copy()
tsla_cls_reset = tsla_cls.reset_index()
tsla_folds = walk_forward_splits(tsla_cls, train_window=59)
print(f'TSLA shape: {tsla_cls.shape}, Folds: {len(tsla_folds)}')

neg = (tsla_cls[TARGET_CLS] == 0).sum()
pos = (tsla_cls[TARGET_CLS] == 1).sum()
tsla_scale_pos_weight = neg / pos
print(f'scale_pos_weight: {tsla_scale_pos_weight:.3f}')

TSLA shape: (2515, 21), Folds: 58
scale_pos_weight: 0.930


In [7]:
tsla_actual, tsla_pred, tsla_seasons = [], [], []
fold_scores_tsla = []

for train_idx, test_idx in tsla_folds:
    X_train = tsla_cls.iloc[train_idx][FEATURES]
    y_train = tsla_cls.iloc[train_idx][TARGET_CLS]
    X_test  = tsla_cls.iloc[test_idx][FEATURES]
    y_test  = tsla_cls.iloc[test_idx][TARGET_CLS]
    seasons = tsla_cls_reset.iloc[test_idx]['season'].values

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    svm = SVC(C=10, gamma=0.1, kernel='rbf', class_weight='balanced', random_state=42, probability=True)
    lr  = LogisticRegression(C=10, penalty='l1', solver='liblinear', class_weight='balanced', max_iter=1000, random_state=42)
    rf  = RandomForestClassifier(n_estimators=200, max_depth=4, min_samples_leaf=10,
                                  max_features='sqrt', class_weight='balanced', random_state=42, n_jobs=-1)
    xgb = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.01, subsample=1.0,
                         scale_pos_weight=tsla_scale_pos_weight, objective='binary:logistic',
                         eval_metric='logloss', random_state=42, n_jobs=-1, verbosity=0)

    svm.fit(X_train_sc, y_train)
    lr.fit(X_train_sc, y_train)
    rf.fit(X_train, y_train)
    xgb.fit(X_train, y_train)

    svm_pred = svm.predict(X_test_sc)
    lr_pred  = lr.predict(X_test_sc)
    rf_pred  = rf.predict(X_test)
    xgb_pred = xgb.predict(X_test)

    stacked = np.stack([svm_pred, lr_pred, rf_pred, xgb_pred], axis=1)
    y_p = (stacked.sum(axis=1) >= 2).astype(int)

    cm = confusion_matrix(y_test, y_p, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0

    fold_scores_tsla.append({
        'accuracy':    accuracy_score(y_test, y_p),
        'f1':          f1_score(y_test, y_p, zero_division=0),
        'precision':   precision_score(y_test, y_p, zero_division=0),
        'recall':      recall_score(y_test, y_p, zero_division=0),
        'specificity': spec
    })
    tsla_actual.extend(y_test)
    tsla_pred.extend(y_p)
    tsla_seasons.extend(seasons)

print(classification_report(tsla_actual, tsla_pred, zero_division=0))
print(pd.DataFrame(fold_scores_tsla).describe())

              precision    recall  f1-score   support

           0       0.48      0.35      0.40      1170
           1       0.52      0.64      0.57      1266

    accuracy                           0.50      2436
   macro avg       0.50      0.50      0.49      2436
weighted avg       0.50      0.50      0.49      2436

        accuracy         f1  precision     recall  specificity
count  58.000000  58.000000  58.000000  58.000000    58.000000
mean    0.502463   0.544564   0.532106   0.653083     0.360871
std     0.081226   0.152870   0.110167   0.270921     0.284555
min     0.333333   0.080000   0.333333   0.043478     0.000000
25%     0.452381   0.480317   0.444444   0.483696     0.137987
50%     0.476190   0.576037   0.517880   0.683333     0.308424
75%     0.547619   0.642204   0.591779   0.897222     0.554167
max     0.690476   0.793651   0.818182   1.000000     0.947368


In [8]:
seasonal_breakdown(tsla_actual, tsla_pred, tsla_seasons)


Seasonal Performance Breakdown:
Season        n     acc      f1    prec     rec
--------------------------------------------------
Winter      548   0.540   0.600   0.559   0.647
Spring      613   0.488   0.534   0.477   0.606
Summer      645   0.481   0.549   0.510   0.595
Fall        630   0.506   0.608   0.525   0.722
